# 02 · Exam windows: when did each student take each real midterm?

Runestone logs a `timedExam` *start* and *finish* marker for every timed assessment. The same mechanism is
used for practice exams and in-class quizzes, so the first job is deciding which names are the real midterms.
The window `[start, finish]` is used twice later: to score the exam, and to keep exam answers out of "practice".

In [1]:
import pandas as pd
from so_ell import config
from so_ell.notebook import show, show_sql, describe
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
config.ensure_dirs()
T = config.TABLES_DIR

## 1. Which timed exams count as a real midterm

In [2]:
from so_ell import exam_windows as ew
show(ew.parse_midterm, ew.is_real_exam_name)
print("excluded words:", ew.NOT_REAL_EXAM_PATTERN)

`so_ell.exam_windows.parse_midterm`
```python
def parse_midterm(name: object) -> str | None:
    cleaned = clean_name(name)
    if re.search(r"\b(?:m|mid|midterm)\s*1\b", cleaned):
        return "mid1"
    if re.search(r"\b(?:m|mid|midterm)\s*2\b", cleaned):
        return "mid2"
    return None
```

`so_ell.exam_windows.is_real_exam_name`
```python
def is_real_exam_name(name: object) -> bool:
    cleaned = clean_name(name)
    if parse_midterm(cleaned) is None:
        return False
    return not re.search(NOT_REAL_EXAM_PATTERN, cleaned)
```

excluded words: \b(practice|post|quiz|study|intro|pi\w*)\b


## 2. Load the start/finish markers (DuckDB, reading only the rows needed)

In [3]:
show(ew.load_timed_markers)
markers = ew.load_timed_markers()
print(f"{len(markers):,} timedExam markers")
markers.groupby(["semester_raw", "action"]).size().unstack()

`so_ell.exam_windows.load_timed_markers`
```python
def load_timed_markers(events_path: Path = config.EVENTS_PARQUET) -> pd.DataFrame:
    con = duckdb.connect()
    con.execute("SET enable_progress_bar = false")
    markers = con.execute(
        f"""
        SELECT time, semester AS semester_raw, anon_student_id, session_id, problem_name, action
        FROM read_parquet('{events_path.as_posix()}')
        WHERE selection = 'timedExam' AND action IN ('start', 'finish') AND time IS NOT NULL
        """
    ).df()
    con.close()
    return markers
```

16,159 timedExam markers


action,finish,start
semester_raw,,
F21,299,355
F22,1334,1554
F23,795,1146
F24,1082,1380
F25,1268,1512
W22,658,753
W23,1009,1351
W24,679,984


## 3. Every timed-exam name, and whether it was kept
This audit table is the evidence for the name rule above.

In [4]:
show(ew.anchor_name_audit)
audit = ew.anchor_name_audit(markers)
audit.to_csv(T / "exam_name_audit.csv", index=False)
audit.sort_values("students", ascending=False).head(30)

`so_ell.exam_windows.anchor_name_audit`
```python
def anchor_name_audit(markers: pd.DataFrame) -> pd.DataFrame:
    """Every timed-exam name with a midterm number, and whether it was kept.

    This is the table to read when someone asks "how do you know which
    timed exam was the real midterm?"
    """
    use = markers.loc[markers["action"].eq("start")].copy()
    use["midterm"] = use["problem_name"].map(parse_midterm)
    use = use.loc[use["midterm"].notna()]
    use["kept_as_real_exam"] = use["problem_name"].map(is_real_exam_name)
    return (
        use.groupby(["semester_raw", "problem_name", "midterm", "kept_as_real_exam"], as_index=False)
        .agg(students=("anon_student_id", "nunique"), first_start=("time", "min"), last_start=("time", "max"))
        .sort_values(["semester_raw", "midterm", "students"], ascending=[True, True, False])
    )
```

,semester_raw,problem_name,midterm,kept_as_real_exam,students,first_start,last_start
67,F24,M2-F24-Timed,mid2,True,213,2024-10-30 20:00:18.198763,2024-10-31 16:20:47.207746
59,F24,M1-F24-80,mid1,True,211,2024-10-02 20:06:10.259784,2024-10-03 15:47:49.849038
86,F25,Mid2-F25-Timed,mid2,True,192,2025-10-29 20:00:03.235881,2025-10-31 02:34:31.726316
36,F23,Mid1-Fall23-Timed,mid1,True,190,2023-10-04 19:00:18.203456,2023-10-09 20:52:04.905127
73,F25,M1-F25,mid1,True,189,2025-10-01 20:00:11.270164,2025-10-02 16:13:31.889511
81,F25,M2-F25-PI-Timed,mid2,False,188,2025-10-27 19:54:02.271593,2025-12-18 12:19:48.661499
126,W24,Mid1-W24-Timed-80,mid1,True,178,2024-02-15 14:39:10.138366,2024-02-18 02:40:17.735482
76,F25,M1-F25-PI-Timed,mid1,False,178,2025-09-29 20:49:51.047133,2025-10-06 18:08:25.239580
105,W23,Mid1-Win23-Timed,mid1,True,177,2023-02-09 15:00:34.208063,2023-02-13 21:10:19.293640
65,F24,M2-F24-PI-Timed,mid2,False,176,2024-10-28 20:06:43.926280,2024-11-05 20:48:52.905740


## 4. Build one attempt per student, session and midterm

In [5]:
show(ew.build_attempts, ew.run_exam_windows)
result = ew.run_exam_windows()
attempts = result["attempts"]
from so_ell import validation
validation.validate(attempts, validation.exam_attempts_schema)
assert result["missing"].empty, result["missing"]
result["summary"].to_csv(T / "exam_window_summary.csv", index=False)
print(f"{len(attempts):,} attempts; every semester has both midterms")
result["summary"]

`so_ell.exam_windows.build_attempts`
```python
def build_attempts(markers: pd.DataFrame) -> pd.DataFrame:
    markers = markers.copy()
    markers["midterm"] = markers["problem_name"].map(parse_midterm)
    markers["is_real_exam"] = markers["problem_name"].map(is_real_exam_name)
    real = markers.loc[markers["midterm"].notna() & markers["is_real_exam"]]

    keys = ["semester_raw", "anon_student_id", "session_id", "midterm"]
    starts = real.loc[real["action"].eq("start")].groupby(keys)["time"].min().rename("attempt_start")
    ends = real.loc[real["action"].eq("finish")].groupby(keys)["time"].max().rename("attempt_end")
    names = real.groupby(keys)["problem_name"].agg(lambda s: " | ".join(sorted(set(s)))).rename("anchor_problem_name")

    attempts = pd.concat([starts, ends, names], axis=1).reset_index()
    attempts = attempts.loc[attempts["attempt_start"].notna()].copy()
    attempts["attempt_end"] = attempts["attempt_end"].fillna(attempts["attempt_start"])
    # A finish logged before the start belongs to an earlier session; ignore it.
    attempts["attempt_end"] = attempts[["attempt_start", "attempt_end"]].max(axis=1)
    attempts["semester"] = attempts["semester_raw"].map(semester_label)
    attempts["normalized_midterm_name"] = attempts["semester"] + " " + attempts["midterm"].map(MIDTERM_LABELS)
    attempts["attempt_minutes"] = (attempts["attempt_end"] - attempts["attempt_start"]).dt.total_seconds() / 60
    return attempts.sort_values(keys).reset_index(drop=True)
```

`so_ell.exam_windows.run_exam_windows`
```python
def run_exam_windows(
    events_path: Path = config.EVENTS_PARQUET,
    out_path: Path = config.EXAM_ATTEMPTS_PARQUET,
) -> dict[str, pd.DataFrame]:
    markers = load_timed_markers(events_path)
    attempts = build_attempts(markers)
    attempts = attempts.loc[attempts["semester_raw"].isin(config.ANALYSIS_SEMESTERS)].copy()
    attempts.to_parquet(out_path, index=False)

    summary = (
        attempts.groupby(["semester_raw", "semester", "midterm"], as_index=False)
        .agg(
            attempts=("session_id", "size"),
            students=("anon_student_id", "nunique"),
            median_minutes=("attempt_minutes", "median"),
            first_start=("attempt_start", "min"),
            last_start=("attempt_start", "max"),
        )
    )
    expected = pd.MultiIndex.from_product(
        [[s for s in config.ANALYSIS_SEMESTERS if s in set(markers["semester_raw"])], ["mid1", "mid2"]],
        names=["semester_raw", "midterm"],
    ).to_frame(index=False)
    missing = expected.merge(summary[["semester_raw", "midterm"]], how="left", indicator=True)
    missing = missing.loc[missing["_merge"].eq("left_only")].drop(columns="_merge")
    return {"attempts": attempts, "summary": summary, "missing": missing, "name_audit": anchor_name_audit(markers)}
```

2,905 attempts; every semester has both midterms


,semester_raw,semester,midterm,attempts,students,median_minutes,first_start,last_start
0,F21,Fall 2021,mid1,104,103,79.308333,2021-10-06 16:57:46.000000,2021-10-07 22:10:32.000000
1,F21,Fall 2021,mid2,103,103,78.866667,2021-11-03 16:58:11.000000,2021-11-08 14:14:50.000000
2,F22,Fall 2022,mid1,149,148,76.523835,2022-10-06 02:27:52.125488,2022-10-10 23:01:51.368787
3,F22,Fall 2022,mid2,146,146,78.398961,2022-11-03 13:59:34.024098,2022-11-07 02:11:58.415098
4,F23,Fall 2023,mid1,207,207,78.778080,2023-10-04 19:00:18.203456,2023-10-09 20:52:04.905127
5,F23,Fall 2023,mid2,202,201,77.540638,2023-10-30 04:29:32.716096,2023-11-09 17:26:29.955216
6,F24,Fall 2024,mid1,224,223,80.092138,2024-10-02 16:56:51.692462,2024-10-03 15:47:49.849038
7,F24,Fall 2024,mid2,222,222,76.411848,2024-10-30 20:00:18.198763,2024-11-04 14:59:33.183539
8,F25,Fall 2025,mid1,214,214,76.047269,2025-10-01 17:50:10.038999,2025-10-03 21:09:50.561964
9,F25,Fall 2025,mid2,213,213,77.401543,2025-10-29 17:39:16.439612,2025-10-31 21:09:46.412109


In [6]:
attempts.groupby("midterm")["attempt_minutes"].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
midterm,,,,,,,,
mid1,1404.0,71.7,20.4,0.0,62.4,77.2,80.2,270.2
mid2,1501.0,72.5,51.4,0.0,62.8,76.4,80.1,901.7


## 5. Why the windows matter for practice features

Count answer events that fall inside a student's **midterm 1** window but before their **midterm 2**.
A naive `time < exam_start` filter would count every one of these as "practice for midterm 2".

In [7]:
import duckdb
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.register("attempts", attempts)
EV = f"read_parquet('{config.EVENTS_PARQUET.as_posix()}')"
con.execute(f'''
WITH w AS (SELECT semester_raw, anon_student_id, midterm, MIN(attempt_start) s, MAX(attempt_end) e FROM attempts GROUP BY ALL)
SELECT COUNT(*) AS mid1_exam_events_before_mid2
FROM {EV} ev
JOIN w m1 ON m1.semester_raw = ev.semester AND m1.anon_student_id = ev.anon_student_id AND m1.midterm = 'mid1'
JOIN w m2 ON m2.semester_raw = ev.semester AND m2.anon_student_id = ev.anon_student_id AND m2.midterm = 'mid2'
WHERE ev.time BETWEEN m1.s AND m1.e AND ev.time < m2.s
''').df()

,mid1_exam_events_before_mid2
0,1254782
